Embedding model used : Huggingface embeddings  
Vector store used : Chromadb

In [1]:
!pip install langchain langchain-community langchain-text-splitters chromadb pypdf

# Part1

# Task1

In [18]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage,SystemMessage,HumanMessage
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough

file_path = "C:/Users/kumar/OneDrive/Desktop/TRY-2/Tasks-Submission/Assignment28/4 - Harry Potter and the Goblet of Fire.pdf"  # or "sample.pdf"

if file_path.endswith(".pdf"):
    loader = PyPDFLoader(file_path)
else:
    loader = TextLoader(file_path, encoding="utf-8")

documents = loader.load()

print("Number of documents:", len(documents))
print("Sample content:\n", documents[0].page_content[:500])


Number of documents: 740
Sample content:
 


# Task2

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))
print("Sample chunk:\n", chunks[0].page_content[:300])


Number of chunks: 2849
Sample chunk:
 Harry Potter
and the Goblet Of Fire
 
 
by
J. K. Rowling
Illustrations by Mary Grandpré
 
 
 
 
Arthur A. Levine Books
An Imprint of Scholastic Press


# Part2

# Task3

In [3]:
!pip install sentence-transformers

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


C:\Windows\Temp\ipykernel_24656\3055314890.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 490.17it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Task4

In [7]:
from langchain_community.vectorstores import Chroma
import uuid

collection_name = f"rag_collection_{uuid.uuid4().hex}"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory="chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


# Part3

# Task5

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer ONLY from the context below.\n"
     "If the answer is not found, say: I don't know.\n\nContext:\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])


# Part4

# Task6

In [19]:
llm = ChatGroq(model='llama-3.1-8b-instant')

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def rag_chain(question, chat_history):
    docs = retriever.invoke(question)
    context = format_docs(docs)

    prompt = rag_prompt.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })

    return llm.invoke(prompt).content


# Task7

In [20]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that rewrites questions using chat history."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])


In [21]:
from langchain_core.runnables import RunnablePassthrough

history_aware_retriever = (
    history_prompt
    | llm
    | StrOutputParser()
    | retriever
)


In [22]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the provided context."),
    ("human", "Context:\n{context}\n\nQuestion:\n{input}")
])


In [23]:
qa_chain = qa_prompt | llm | StrOutputParser()
chat_history = []

# Task8

In [28]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [
    HumanMessage(content="Who is Harry Potter ?"),
    AIMessage(content="Harry potter is the main character in the story ..")
]

question = "Who is ALbus "

docs = history_aware_retriever.invoke({
    "input": question,
    "chat_history": chat_history
})

context = "\n\n".join(doc.page_content for doc in docs)

answer = qa_chain.invoke({
    "input": question,
    "context": context
})

print(answer)


Albus is Dumbledore.


In [29]:
def trim_chat_history(chat_history, max_turns=4):
    """
    Keeps only the last `max_turns` user-assistant pairs.
    
    chat_history format:
    [
        ("user", "message"),
        ("assistant", "reply"),
        ...
    ]
    """
    max_messages = max_turns * 2
    return chat_history[-max_messages:]


# Part5

# Task9

In [30]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [
    HumanMessage(content="Tell me about Harry Potter"),
    AIMessage(content="Harry potter is a main character .. ")
]

question = "Who was Professor DumbleDore ? and his role?"

docs = history_aware_retriever.invoke({
    "input": question,
    "chat_history": chat_history
})

context = "\n\n".join(doc.page_content for doc in docs)

answer = qa_chain.invoke({
    "input": question,
    "context": context
})

print(answer)


The passage does not explicitly mention Professor Dumbledore's role as "Professor", but it mentions his title as "Headmaster of Hogwarts" which suggests that he was the head of the school.


# Part6

# Task10

In [27]:
def ask(question, chat_history):
    # 1. Retrieve docs using history
    docs = history_aware_retriever.invoke({
        "input": question,
        "chat_history": chat_history
    })

    print("\n--- Retrieved context ---")
    for d in docs:
        print("-", d.page_content[:120], "...")

    context = "\n\n".join(d.page_content for d in docs)

    # 2. Generate answer using context
    answer = qa_chain.invoke({
        "input": question,
        "context": context
    })

    # 3. Update history
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    return answer

In [31]:
q1 = "Who is Professor DumbleDore"
a1 = ask(q1, chat_history)

print("\nAnswer 1:", a1)


--- Retrieved context ---
- The Daily Prophet, however, has unearthed worrying facts
about Harry Potter that Albus Dumbledore, Headmaster of
Hogwart ...
- guessed was Professor McGonagall’s. Next to it, and in the
very center of the table, sat Professor Dumbledore, the
headm ...
- teachers.
They had never yet had a Defense Against the Dark Arts
teacher who had lasted more than three terms. Harry’s
f ...
- thoughtfully at Ron and Hermione. “Ever since I found out
Snape was teaching here, I’ve wondered why Dumbledore
hired hi ...

Answer 1: Professor Dumbledore is the Headmaster of Hogwarts.


In [33]:
q2 = "What was his appearance ?"
a2 = ask(q1, chat_history)

print("\nAnswer 2:", a2)


--- Retrieved context ---
- guessed was Professor McGonagall’s. Next to it, and in the
very center of the table, sat Professor Dumbledore, the
headm ...
- blaming Dumbledore for Potter’s determination to break
rules. He has been crossing lines ever since he arrived here
—”
“ ...
- “SLYTHERIN!”
“Quirke, Orla!”
“RAVENCLAW!”
And ﬁnally, with “Whitby, Kevin!” (“HUFFLEPUFF!”), the
Sorting ended. Professo ...
- they were wearing cloaks of some kind of shaggy, matted
fur. But the man who was leading them up to the castle was
weari ...

Answer 2: Professor Dumbledore's appearance is described as follows:
- He had sweeping silver hair and a beard.
- His robes were magnificent, deep green, and embroidered with many stars and moons.
- He wore half-moon spectacles.
- He has long, thin fingers.


In [34]:
q3 = "Can you generate the summary of the book ?"
a3 = ask(q3, chat_history)

print("\nAnswer 3:", a3)


--- Retrieved context ---
- York, NY 10012.
 
Library of Congress Cataloging-in-Publication Data
Available
 
Library of Congress catalog card number ...
- books she had, “It’s all in Hogwarts, A History. Though, of
course, that book’s not entirely reliable. A Revised History ...
- Harry Potter
and the Goblet Of Fire
 
 
by
J. K. Rowling
Illustrations by Mary Grandpré
 
 
 
 
Arthur A. Levine Books
A ...
- Text copyright © 2000 by J.K. Rowling
Illustrations by Mary GrandPre copyright © 2000 Warner
Bros.
All rights reserved.  ...

Answer 3: Fourteen-year-old Harry Potter joins the Weasleys at the Quidditch World Cup, then enters his fourth year at Hogwarts School of Witchcraft and Wizardry.


In [36]:
q4 = "What's his age ?"
a4 = ask(q4, chat_history)

print("\nAnswer 4:", a4)


--- Retrieved context ---
- York, NY 10012.
 
Library of Congress Cataloging-in-Publication Data
Available
 
Library of Congress catalog card number ...
- found what they were looking for. Harry and Ron leaned in
closer. A color photograph of Harry headed a short piece
entit ...
- “Speak for yourself,” said George shortly. “You’ll try and
get in, won’t you, Harry?”
Harry thought brieﬂy of Dumbledore ...
- Harry has at last found love at Hogwarts. His close
friend, Colin Creevey, says that Harry is rarely seen out of
the com ...

Answer 4: Fourteen.


# Task11

1. Difference between PDF Q&A and Conversational PDF Q&A
PDF Q&A is a single-turn, stateless system where each question is answered independently using only the document content, making it fast and cost-effective but weak for follow-up questions. Conversational PDF Q&A, on the other hand, maintains context across multiple turns by using conversation history, allowing it to understand references, pronouns, and follow-up queries, which results in a more natural and intelligent user experience at the cost of higher computation and latency.

2. Role of message history in follow-up questions
Message history provides contextual continuity by allowing the model to understand references to earlier questions and answers, such as pronouns or implicit mentions like “this” or “the second one.” It enables coherent multi-turn conversations by preserving the user’s intent and topic flow, ensuring that follow-up questions are interpreted correctly without requiring the user to restate information.

3. Trade-offs between long memory and performance
Maintaining long conversational memory improves contextual understanding and answer coherence but increases token usage, latency, and computational cost, which can reduce system scalability. Shorter memory improves speed and efficiency but may lose important context, leading to weaker follow-up responses, making it necessary to balance memory length with system performance.

4. How trimming history affects answer quality
Trimming conversation history can improve response speed and reduce noise in the prompt, leading to clearer and more focused answers, but excessive trimming may remove critical context needed for accurate follow-up responses. The best approach is selective trimming or summarization, which preserves essential information while discarding irrelevant or resolved interactions to maintain answer quality.